# Data wrangling for DORA Metrics

## Env config

Test kernel

In [ ]:
print("Kernel is working.")

Environment config

In [ ]:
from dash import dcc
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

## Deployments Data Wrangling
Outputs:

- dataframe: deployments (has new columns for quatetr and month)

### Import data into df from file.

In [ ]:
# Useful reference: <https://pandas.pydata.org/docs/getting_started/intro_tutorials/09_timeseries.html>
raw_file_path = '/home/lnx_workspaces/bpp_projects/bpp_module2_project/doraview/data/json/deployment_frequency.json'

# Try reading with explicit encoding and error handling
try:
    deployments = pd.read_json(raw_file_path, encoding='utf-8', convert_dates=["deployed_at"])
    print("\nDataframe info:")
    print(deployments.info())
    print("\nFirst 5 rows:")
    print(deployments.head())
except Exception as e:
    print(f"Error: {str(e)}")

### Add new sorting columns

Add new columns "Quater", "Month", "Week" to show which of these each deployment occured in.

#### New columns

In [ ]:
deployments["qtr"] = deployments["deployed_at"].dt.quarter
deployments["month"] = deployments["deployed_at"].dt.month

deployments.head(100)

#### Clean table

Clean to only the columns I want.

In [ ]:
deploys = deployments[["application_id","application_name","environment","status","month"]]
deploys.head(5)

### Creating the long form table

Groupings by:

- environment
- status
- month

#### Create heirarchical groups.

In [ ]:
# This pre-selects the columns of interest, app_id and month. So only these show.
deploys[["application_id","month"]].groupby("month").count()

#### Ful split of table

In [ ]:
# This method is not dictating which columns are selected, only those that the data is grouped by, and then which count.
split_df = deploys.groupby(["application_id","environment","status","month"])["month"].count().reset_index(name="count")
split_df.head(100)

#### df for Plot

- **df/plot 1:** For each month, count of deployments for each app.
- **df/plot 2:** For one app, count dep each env
- **df/plot 3:** For one app, count dep succ/fail

##### df

In [ ]:
# df 1
deploys.groupby(["application_id","month"])["month"].count().reset_index(name="count")

##### plot

In [ ]:
fig_app_test = px.bar(split_df, x="month", y="count", color="application_id", title="Long-Form Input")
fig_app_test.update_yaxes(title_text="Deployment Count")
fig_app_test.update_xaxes(title_text="Deployment Month", tickvals=list(range(1,13)), ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])

fig_app_test.show()

Staging deployments

In [ ]:
staging = deployments[deployments["environment"]=="staging"]

staging.head(5)

app2 deployments in prod

In [ ]:
# ap2 = df[(df["environment"]=="production") & (df["application_id"]=="app002")]
ap2 = deployments[(deployments["application_id"]=="app002")]

ap2.head(5)

In [ ]:
deployments["application_id"].value_counts()

In [ ]:
deployments["deployed_at"].min(), deployments["deployed_at"].max()

In [ ]:
deployments["deployed_at"].max() - deployments["deployed_at"].min()

## Plotting

### Plots with matlibplot

In [ ]:
count_deployments = deployments.groupby(deployments["deployed_at"].dt.month)["application_id"].count()
count_deployments.head()

In [ ]:
fig, axs = plt.subplots(figsize=(12,4))

# deployments.groupby(deployments["deployed_at"].dt.month)["application_id"].count().plot(kind="bar", rot=0, ax=axs)
count_deployments.plot(kind="bar", rot=0, ax=axs)

### Plots with plotly express

#### Plotly Express example data

In [ ]:
px_gap_data = px.data.gapminder()

px_gap_data.head()

In [ ]:
px_medal_data = px.data.medals_long()
px_medal_data.head()

In [ ]:
fig_medals = px.bar(px_medal_data, x="nation", y="count", color="medal", title="Long-Form Input")
fig_medals.show()

In [ ]:
px_medal_wide_data = px.data.medals_wide()

px_medal_wide_data.head()

In [ ]:
px_tips_data = px.data.tips()
px_tips_data.head()

#### My work

Using Plotly Express

In [ ]:
deployments.head()

In [ ]:
# # This method is not dictating which columns are selected, only those that the data is grouped by, and then which count.
# split_df = deploys.groupby(["application_id","environment","status","month"])["month"].count().reset_index(name="count")

deployments.groupby(["month","environment"])["month"].count()

In [ ]:
fig_deploy = px.bar(deployments, x="application_id", y="month", color="status", title="Long-Form Input")
fig_deploy.show()

In [ ]:
monthly_counts = deployments.groupby(deployments["deployed_at"].dt.month)["application_id"].count().reset_index()

monthly_counts.head()

In [ ]:
fig = px.bar(monthly_counts, x='deployed_at', y='application_id', title='Total Application Deployments', )
fig.update_yaxes(title_text="Deployment Count")
fig.update_xaxes(title_text="Deployment Month", tickvals=list(range(1,13)), ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
fig.show()

## Messing with data